# EXP-028 v2 — критик на оценках закрытой части ответа

Проверяю вариант Г для PPO. Беру 640 уже собранных ответов сида 0, считаю признаки закрытых слайдов и записываю время по ходу расчёта. Затем проверяю Z-lin на тех же отложенных темах, что в исходном EXP-028.

Для каждого действия использую текст, доступный перед его токеном. Слайд входит в оценку после появления следующей завершённой строки `#SLIDE`. Цель головы — сохранённая итоговая награда полного эпизода.

## 1. Настройки и данные

Читаю `episodes.jsonl` из разогрева сида 0. Токенизатор и лимиты беру из конфига PPO. Проверяю, что доступен весь разогрев. Кеш признаков привязываю к входным данным, ревизии токенизатора и коду оценки.

In [ ]:
import statistics
import sys
import time
from dataclasses import replace
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from presentation_rl.tasks import load_tasks
from presentation_rl.training.common import RewardModel, file_hash, stable_seed, write_json
from presentation_rl.training.ppo import PPOConfig, exp028_verdict, load_warmup_episodes, split_topics
from presentation_rl.training.prefix_features import FEATURE_NAMES, PrefixFeatureExtractor, fit_prefix_head, prepare_prefix_warmup

SEED = 0
WARMUP = ROOT / "runs" / "ppo" / f"warmup_seed{SEED}"
EPISODES = WARMUP / "episodes.jsonl"
FEATURES = WARMUP / "prefix_features.pt"
REPORT = WARMUP / "exp028_v2.json"
TASKS = ROOT / "data" / "tasks" / "tasks.json"
POINTS = ROOT / "data" / "tasks" / "key_points" / "key_points.json"

config = PPOConfig.load(ROOT / "configs" / "algorithms" / "ppo.yaml", seed=SEED, warmup_dir=str(WARMUP))
episodes = load_warmup_episodes(WARMUP)
per_step = 2 * config.tasks_per_mode * config.group_size
expected = config.warmup_steps * per_step
if len(episodes) != expected:
    raise ValueError(f"В {EPISODES} {len(episodes)} эпизодов, ожидаю {expected}")
tasks = {task.id: task for task in load_tasks(TASKS)}

SRC = ROOT / "src" / "presentation_rl"
score_files = ("dsl.py", "environment.py", "perturb.py", "rewards.py", "key_points.py",
               "rendering/builder.py", "training/common.py", "training/ppo.py", "training/prefix_features.py")
provenance = {
    "episodes": file_hash(EPISODES), "tasks": file_hash(TASKS), "points": file_hash(POINTS),
    "code": {name: file_hash(SRC / name) for name in score_files},
    "model": config.policy_model, "revision": config.policy_revision,
    "max_new_tokens": config.max_new_tokens, "max_context": config.max_context,
    "reward_floor": config.reward_floor, "max_positions": config.warmup_max_positions,
    "embedder_device": config.embedder_device, "seed": SEED, "feature_names": list(FEATURE_NAMES),
}
print(f"Разогрев: {len(episodes)} эпизодов, {config.warmup_steps} пакетов по {per_step}.")

## 2. Признаки и время расчёта

Вычисляю признаки всего разогрева. Для закрытой части каждого ответа собираю презентацию в памяти и считаю компоненты оценки. В буфер беру выборку действий по тому же правилу, что в исходном EXP-028. Первое и последнее действия сохраняю всегда.

Печатаю время каждого пакета. Загрузку токенизатора и bge-m3 и запись буфера показываю отдельно. При повторном запуске я читаю готовый буфер, если входы совпадают.

Среднее время пакета использую для предварительной оценки расходов на два сида PPO. Фактическую добавку уточню по пилоту: в обучении меняются длины ответов и состояние кеша эмбеддингов.

In [ ]:
cached = torch.load(FEATURES, map_location="cpu", weights_only=True) if FEATURES.exists() else None
if cached and cached.get("provenance") == provenance:
    data = cached["data"]
    print(f"Читаю готовые признаки из {FEATURES.name}. Ниже время их исходного расчёта.")
else:
    from transformers import AutoTokenizer

    began = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(config.policy_model, revision=config.policy_revision)
    reward = RewardModel(ROOT, device=config.embedder_device, floor=config.reward_floor)
    extract = PrefixFeatureExtractor(
        tokenizer, reward, max_new_tokens=config.max_new_tokens, max_context=config.max_context,
    )
    print(f"Загрузка токенизатора и bge-m3: {time.perf_counter() - began:.1f} с")

    began = time.perf_counter()
    data = prepare_prefix_warmup(episodes, tasks, extract, config)
    tmp = FEATURES.with_suffix(".pt.tmp")
    torch.save({"provenance": provenance, "data": data}, tmp)
    tmp.replace(FEATURES)
    print(f"Расчёт и сохранение: {(time.perf_counter() - began) / 60:.1f} мин · {FEATURES.name}")
    del extract, reward, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

times = [row["seconds"] for row in data["timing"]]
step_seconds = statistics.fmean(times)
step_scores = statistics.fmean(row["score_calls"] for row in data["timing"])
print(f"Буфер: {len(data['features'])} эпизодов · {sum(map(len, data['features']))} позиций · "
      f"{len(data['feature_names'])} признаков")
print(f"Извлечение на разогреве: {sum(times) / 60:.1f} мин, самый долгий пакет {max(times):.1f} с")
print(f"Добавка на шаг PPO (оценка): {step_seconds:.1f} с · около {step_scores:.0f} оценок")
print(f"Добавка на два сида по {config.steps} шагов: {2 * config.steps * step_seconds / 3600:.2f} ч")

## 3. Проверка Z-lin

Делю эпизоды по темам с тем же сидом, что в первом EXP-028. Стандартизацию признаков, общую константу и средние награды по режимам считаю на обучающих темах. Обучаю линейную голову с `lr = 1e-3` на CPU. Как в исходном EXP-028, эпоху выбираю по MSE на отложенной части.

Для обеих точек сравнения показываю парную разность ошибок с бутстрепом по темам. Положительная разность означает меньшую ошибку Z-lin. Поскольку эти темы уже использовались в первом EXP-028 и для выбора эпохи, читаю интервалы как результат на наборе разработки. Вердикт исходного EXP-028 считаю по общей константе. Сравнение со средними по режимам показываю рядом.

In [ ]:
holdout = split_topics(data["tasks"], config.warmup_holdout, stable_seed(SEED, "warmup-split"))
began = time.perf_counter()
fit = fit_prefix_head(data, replace(config, head_device="cpu"), holdout, lr=1e-3)
fit_seconds = time.perf_counter() - began
metrics = fit["holdout"]
verdict = exp028_verdict(metrics)

print(f"Отложено тем: {len(holdout)} из {len(set(data['tasks']))} · "
      f"лучшая эпоха {fit['best_epoch']} · обучение и оценка {fit_seconds:.1f} с")
for label, metric in (("Общая константа", "paired"), ("Среднее по режиму", "paired_mode")):
    p = metrics[metric]
    print(f"{label} · парная мера {p['estimate']:+.5f} [{p['low']:+.5f}; {p['high']:+.5f}]")
for scope in ("all", "plain_all", "search_all"):
    if scope + "_mse" in metrics:
        print(f"{scope} · MSE {metrics[scope + '_mse']:.5f} | "
              f"константа {metrics[scope + '_mse_const']:.5f} | режим {metrics[scope + '_mse_mode']:.5f}")
print(f"Смещение V − R: {metrics['all_bias']:+.5f}")
print("Вердикт по исходному правилу EXP-028:", verdict)

write_json(REPORT, {
    "seed": SEED, "holdout_topics": sorted(holdout), "provenance": provenance,
    "timing": data["timing"], "fit_seconds": fit_seconds, "verdict": verdict,
    "fit": {name: value for name, value in fit.items() if name not in ("head", "optimizer")},
})
print(REPORT)